# Why don't all satellites measure hyperspectral radiances at meter scales?

#### 3 case studies:
(1) stationary satellite staring at <b>1 m<sup>2</sup></b> for <b>1 s</b><br>
(2) moving satellite staring at <b>1 m<sup>2</sup></b><br>
(3) moving satellite scanning side-to-side<br><br>

#### what we will (hopefully) learn:
(A) how many photons leave <b>1 m<sup>2</sup></b> of ocean surface<br>
(B) how many photons from this patch reach the satellite detector<br>
(C) how many photons the detector must collect to achieve useful SNR

<div class="alert alert-info" role="alert" style="text-align: center">

## <span style='color:blue'>the challenge:</span> achieve an SNR of 1000 at 678 nm

SNR defined (via oversimplification) as N<sub>electrons</sub> / sqrt(N<sub>electrons</sub>)<br>
Assumes no dark current or noise

</div>

In [33]:
import numpy as np

## <span style='color:blue'>Key characteristics of the observatory and instrument</span>

<img src='satellite_geometry_instrument_char.png' width=1000 alt='Figure depicting satellite viewing geometries'>

In [34]:
altitude = 705000                               # m
tilt = 20                                       # deg
slant = altitude / np.cos(np.deg2rad(tilt))     # m
velocity = 6838                                 # m/s

OE = 0.6                                        # unitless
QE = 0.9                                        # unitless
aperature = 0.09                                # m
omega = np.pi * aperature**2 / slant**2         # sr

wl = 0.678                                      # um
bandwidth = 0.01                                # um
TOA = 14.5                                      # W/m2/um/sr
ocean_mult = 0.05                               # unitless


## <span style='color:blue'>Governing equations</span>

### Power reaching the detector:
<img src=power_reaching_detector.png height=80 alt="Equation describing the power reaching the detector">

### Photoelectrons reaching the detector:
<img src=photons_reaching_detector.png  height=80 alt="Equation describing the number of photoelectrons reaching the detector">

In [35]:
def calculate_SNR(TOA,omega,width,OE,bandwidth,time,QE,wl,ocean_mult):

    h = 6.63e-34                                            # Planck's constant (J/s)
    c = 3e14                                                # speed of sound (m/s)

    P_detector = TOA * omega * width**2 * OE * bandwidth    # Watts

    N_electrons = P_detector * time * QE * wl / h / c       # unitless

    N_ocean = N_electrons * ocean_mult                      # unitless

    SNR = N_electrons / np.sqrt(N_electrons)                # unitless

    print(f"Power at detector: {P_detector:.3e} (Watts)")
    print("Photoelectrons reaching detector:", round(N_electrons))
    print("Photoelectrons from the ocean reaching the detector:", round(N_ocean))
    print("SNR for this scenario:", round(SNR))

## <span style='color:blue'>Case 1: a stationary satellite taking a quick peek at Earth</span>

### <b>1 m<sup>2</sup></b> areal footprint and <b>1 s</b> integration time

In [36]:
width = 1
time = 1

SNR_case1 = calculate_SNR(TOA,omega,width,OE,bandwidth,time,QE,wl,ocean_mult)

Power at detector: 3.933e-15 (Watts)
Photoelectrons reaching detector: 12067
Photoelectrons from the ocean reaching the detector: 603
SNR for this scenario: 110


### Q: what integration time raises the SNR to > 1000?
### A: 90 s

In [37]:
width = 1
time = 90

SNR_case1a = calculate_SNR(TOA,omega,width,OE,bandwidth,time,QE,wl,ocean_mult)

Power at detector: 3.933e-15 (Watts)
Photoelectrons reaching detector: 1085994
Photoelectrons from the ocean reaching the detector: 54300
SNR for this scenario: 1042


## <span style='color:blue'>Case 2: a moving satellite taking a quick peek at Earth</span>

### <b>1 m<sup>2</sup></b> areal footprint and <b>0.000146 s</b> integration time

In [38]:
width = 1
time = width / velocity

print(f"Integration time: {time:.3e} (s)")

Integration time: 1.462e-04 (s)


In [39]:
void = calculate_SNR(TOA,omega,width,OE,bandwidth,time,QE,wl,ocean_mult)

Power at detector: 3.933e-15 (Watts)
Photoelectrons reaching detector: 2
Photoelectrons from the ocean reaching the detector: 0
SNR for this scenario: 1


### Q: what if we increase the spatial footprint to <b>1 km<sup>2</sup></b>?

### A: integration time increases by 3 orders of magnitude and area increases by 6 orders of magnitude

In [40]:
width = 1000
time = width / velocity

print(f"Integration time: {time:.3e} (s)")

Integration time: 1.462e-01 (s)


In [41]:
void = calculate_SNR(TOA,omega,width,OE,bandwidth,time,QE,wl,ocean_mult)

Power at detector: 3.933e-09 (Watts)
Photoelectrons reaching detector: 1764638974
Photoelectrons from the ocean reaching the detector: 88231949
SNR for this scenario: 42008


### So ... 1 km<sup>2</sup> is more than enough in this scenario

### Q: why are pushbroom instruments so attractive?

### A: they stare at the ground longer than whiskbroom instruments

In [42]:
width = 100                 # 100 m instead of 1000 m 
time = width / velocity

void = calculate_SNR(TOA,omega,width,OE,bandwidth,time,QE,wl,ocean_mult)

Power at detector: 3.933e-11 (Watts)
Photoelectrons reaching detector: 1764639
Photoelectrons from the ocean reaching the detector: 88232
SNR for this scenario: 1328


## <span style='color:blue'>Case 3: a moving satellite that scans from side-to-side</span>

### keep the <b>1 km<sup>2</sup></b> pixel, but rotate the telescope at <b>1 Hz</b>
### <span style='color:green'>important: the view of Earth is ~ 1/3 of the full 360-deg scan (~= 2 radians)

#### what is the instantaneous field of view (IFOV) in radians (~= pixel size / altitude)?

In [43]:
width = 1000
IFOV = width / slant

print(f"Instantaneous field of view: {IFOV:.3e} (rad)")

Instantaneous field of view: 1.333e-03 (rad)


#### how many pixels in the Earth view swath (~= swath width / IFOV)?

In [44]:
N_swath = 2 / IFOV

print("Number of pixels in a 2 radian wide swath: ", round(N_swath))

Number of pixels in a 2 radian wide swath:  1500


#### recall that 88M photoelectrons reach the detector for a 1 s stare at 1 km<sup>2</sup>
#### how many reach each pixel in our scanning instrument?

In [45]:
width = 1000
time = width / velocity * 0.33 / N_swath

void = calculate_SNR(TOA,omega,width,OE,bandwidth,time,QE,wl,ocean_mult)

Power at detector: 3.933e-09 (Watts)
Photoelectrons reaching detector: 388094
Photoelectrons from the ocean reaching the detector: 19405
SNR for this scenario: 623


#### but SeaWiFS, PACE OCI, others all rotate at 6 Hz?

In [46]:
width = 1000
time = width / velocity * 0.33 / N_swath / 6

void = calculate_SNR(TOA,omega,width,OE,bandwidth,time,QE,wl,ocean_mult)

Power at detector: 3.933e-09 (Watts)
Photoelectrons reaching detector: 64682
Photoelectrons from the ocean reaching the detector: 3234
SNR for this scenario: 254


#### what to do? raise to 4 km<sup>2</sup>?

In [47]:
width = 2000
IFOV = width / slant
print(f"Instantaneous field of view: {IFOV:.3e} (rad)")

N_swath = 2 / IFOV
print("Number of pixels in a 2 radian wide swath: ", round(N_swath))

time = width / velocity * 0.33 / N_swath / 6

void = calculate_SNR(TOA,omega,width,OE,bandwidth,time,QE,wl,ocean_mult)

Instantaneous field of view: 2.666e-03 (rad)
Number of pixels in a 2 radian wide swath:  750
Power at detector: 1.573e-08 (Watts)
Photoelectrons reaching detector: 1034916
Photoelectrons from the ocean reaching the detector: 51746
SNR for this scenario: 1017
